In [2]:
import numpy as np

# vggt_info = np.load("/workspace/data/vggt_npy/run_3.npz", allow_pickle=True)
vggt_info = np.load("/workspace/data/vggt_npy/pro_1/frame_0000_predictions.npz", allow_pickle=True)

In [3]:
vggt_info.keys()

KeysView(NpzFile '/workspace/data/vggt_npy/pro_1/frame_0000_predictions.npz' with keys: pose_enc, pose_enc_list, depth, depth_conf, world_points...)

In [ ]:
for k, v in vggt_info.items():
	print(k, type(v), v.shape if isinstance(v, np.ndarray) else '')

pose_enc <class 'numpy.ndarray'> (2, 9)
pose_enc_list <class 'numpy.ndarray'> ()
depth <class 'numpy.ndarray'> (2, 294, 518, 1)
depth_conf <class 'numpy.ndarray'> (2, 294, 518)
world_points <class 'numpy.ndarray'> (2, 294, 518, 3)
world_points_conf <class 'numpy.ndarray'> (2, 294, 518)
images <class 'numpy.ndarray'> (2, 3, 294, 518)
extrinsic <class 'numpy.ndarray'> (2, 3, 4)
intrinsic <class 'numpy.ndarray'> (2, 3, 3)
world_points_from_depth <class 'numpy.ndarray'> (2, 294, 518, 3)


In [9]:
# ---------- VGGT Point Cloud 可视化 ----------
# world_points: (2, H, W, 3) — 两个视角在 VGGT world 坐标系下的点云
# world_points_conf: (2, H, W) — 置信度，用于过滤噪点
# images: (2, 3, H, W) — RGB 图像，用于点云着色

import numpy as np
import plotly.graph_objects as go

# ---- 参数 ----
conf_threshold = 0.5          # 置信度阈值（0~1），越高点越少但越干净
max_points_per_view = 30000   # 每个视角最多显示多少点（随机采样，避免卡顿）
point_size = 1.5              # 点大小
rng_seed = 42                 # 固定随机种子，便于复现采样结果

rng = np.random.default_rng(rng_seed)

d = vggt_info
world_pts = d["world_points"]        # (2, H, W, 3)
world_conf = d["world_points_conf"]  # (2, H, W)
images = d["images"]                 # (2, 3, H, W), float32 in [0,1]

assert world_pts.ndim == 4 and world_pts.shape[-1] == 3, f"world_points shape 异常: {world_pts.shape}"
assert world_conf.shape == world_pts.shape[:3], f"world_points_conf shape 异常: {world_conf.shape}"
assert images.shape[0] == world_pts.shape[0] and images.shape[1] == 3, f"images shape 异常: {images.shape}"

fig_pc = go.Figure()
view_colors_fallback = ["dodgerblue", "darkorange"]

for v in range(world_pts.shape[0]):
    pts_v = world_pts[v]      # (H, W, 3)
    conf_v = world_conf[v]    # (H, W)
    img_v = images[v]         # (3, H, W)

    # 展平
    H, W = conf_v.shape
    pts_flat = pts_v.reshape(-1, 3)
    conf_flat = conf_v.reshape(-1)

    # 颜色: 转为 (H*W, 3) RGB [0,255]
    rgb_flat = (img_v.transpose(1, 2, 0).reshape(-1, 3) * 255).clip(0, 255).astype(np.uint8)

    # 置信度过滤
    mask = conf_flat > conf_threshold
    pts_ok = pts_flat[mask]
    rgb_ok = rgb_flat[mask]

    # 随机降采样
    if len(pts_ok) > max_points_per_view:
        idx = rng.choice(len(pts_ok), max_points_per_view, replace=False)
        pts_ok = pts_ok[idx]
        rgb_ok = rgb_ok[idx]

    # 构造颜色字符串
    colors_str = [f"rgb({r},{g},{b})" for r, g, b in rgb_ok]

    fig_pc.add_trace(go.Scatter3d(
        x=pts_ok[:, 0], y=pts_ok[:, 1], z=pts_ok[:, 2],
        mode="markers",
        marker=dict(size=point_size, color=colors_str, opacity=0.8),
        name=f"view {v} point cloud ({len(pts_ok)} pts)",
    ))

# 叠加相机位置（用 VGGT extrinsic 算相机中心）
ext = d["extrinsic"]  # (2, 3, 4)
R_ext = ext[:, :3, :3]  # (2,3,3)
t_ext = ext[:, :3, 3]   # (2,3)
cam_centers = -np.einsum("vij,vj->vi", R_ext.transpose(0, 2, 1), t_ext)  # (2,3)

for v, (cc, col) in enumerate(zip(cam_centers, view_colors_fallback)):
    fig_pc.add_trace(go.Scatter3d(
        x=[cc[0]], y=[cc[1]], z=[cc[2]],
        mode="markers+text",
        marker=dict(size=8, color=col, symbol="diamond"),
        text=[f"cam {v}"],
        textposition="top center",
        name=f"cam {v}",
    ))

file_tag = getattr(d, "filename", "vggt_info")
fig_pc.update_layout(
    title=f"VGGT Point Cloud — {file_tag.split('/')[-1]} (conf>{conf_threshold})",
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
        aspectmode="data",
        bgcolor="rgb(20,20,20)",
        xaxis=dict(backgroundcolor="rgb(20,20,20)", gridcolor="gray"),
        yaxis=dict(backgroundcolor="rgb(20,20,20)", gridcolor="gray"),
        zaxis=dict(backgroundcolor="rgb(20,20,20)", gridcolor="gray"),
    ),
    paper_bgcolor="rgb(30,30,30)",
    font=dict(color="white"),
    legend=dict(itemsizing="constant"),
    margin=dict(l=0, r=0, b=0, t=40),
    width=950,
    height=750,
)
fig_pc.show()

print("Done.")
print(f"world_points shape: {world_pts.shape}")
print(f"camera centers (world):\n{cam_centers}")

Done.
world_points shape: (2, 294, 518, 3)
camera centers (world):
[[ 2.6465961e-05 -3.0960695e-05  1.3073164e-04]
 [-8.1124753e-03 -2.3292806e-02 -6.4211395e-03]]
